<a href="https://colab.research.google.com/github/srushtinagaraju/Machine_Learning_1BM24CS424/blob/main/1BM24CS424-Lab-10%20PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
df = pd.read_csv("heart (1) - heart (1).csv")

Saving heart (1) - heart (1).csv to heart (1) - heart (1) (1).csv


In [4]:
import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score


# -------------------------------
# 1. Load CSV file automatically
# -------------------------------
files = [f for f in os.listdir() if f.endswith('.csv')]
print("CSV files found:", files)

df = pd.read_csv(files[0])   # loads first CSV
print("\nDataset Loaded Successfully!\n")
print(df.head())


# -------------------------------
# 2. Identify columns
# -------------------------------
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(exclude=['object']).columns.tolist()

target_col = df.columns[-1]

if target_col in numerical_cols:
    numerical_cols.remove(target_col)

print("\nCategorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)
print("Target Column:", target_col)


# -------------------------------
# 3. Preprocessing (Encoding + Scaling)
# -------------------------------
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(drop='first'), categorical_cols)
])


# -------------------------------
# 4. Split dataset
# -------------------------------
X = df.drop(columns=[target_col])
y = df[target_col]

if y.dtype == 'object':
    y = LabelEncoder().fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# -------------------------------
# 5. Train models WITHOUT PCA
# -------------------------------
models = {
    "SVM": SVC(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier()
}

print("\n===== WITHOUT PCA =====")
results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print(f"{name} Accuracy: {acc:.4f}")


# -------------------------------
# 6. Train models WITH PCA
# -------------------------------
print("\n===== WITH PCA =====")
pca_results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    pca_results[name] = acc

    print(f"{name} Accuracy (PCA): {acc:.4f}")


# -------------------------------
# 7. Final Comparison
# -------------------------------
print("\n===== FINAL COMPARISON =====")

for name in models:
    print(f"{name}: Without PCA = {results[name]:.4f}, With PCA = {pca_results[name]:.4f}")

CSV files found: ['heart (1) - heart (1) (1).csv', 'heart (1) - heart (1).csv']

Dataset Loaded Successfully!

   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA        140          289          0     Normal    172   
1   49   F           NAP        160          180          0     Normal    156   
2   37   M           ATA        130          283          0         ST     98   
3   48   F           ASY        138          214          0     Normal    108   
4   54   M           NAP        150          195          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  HeartDisease  
0              N      0.0       Up             0  
1              N      1.0     Flat             1  
2              N      0.0       Up             0  
3              Y      1.5     Flat             1  
4              N      0.0       Up             0  

Categorical Columns: ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
Numer